In [112]:
from pyspark.sql.functions import *

fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

In [113]:
daily_revenue = (
    fact_orders
    .groupBy("order_date")
    .agg(
        count("order_id").alias("orders"),
        sum("sales_amount").alias("revenue"),
        sum("sales_qty").alias("quantity")
    )
    .orderBy("order_date")
)

In [114]:
display(daily_revenue)

In [115]:
daily_revenue.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/forecast_base"
    )

In [116]:
monthly_revenue = (
    fact_orders
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month")
    )
    .agg(
        sum("sales_amount").alias("revenue"),
        count("order_id").alias("orders")
    )
    .orderBy("year", "month")
)

In [117]:
display(monthly_revenue)

In [118]:
monthly_revenue.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/monthly_forecast_base"
    )

In [119]:
from pyspark.sql.window import Window

window_spec = Window.orderBy("order_date").rowsBetween(-7, 0)

forecast_dataset = (
    daily_revenue
    .withColumn(
        "forecast_revenue",
        avg("revenue").over(window_spec)
    )
    .withColumn(
        "forecast_orders",
        avg("orders").over(window_spec)
    )
)

In [120]:
forecast_dataset.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/revenue_forecast"
    )

In [121]:
display(
    forecast_dataset.orderBy("order_date")
)